In [179]:
import json
import os
import time
import joblib
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from sklearn.neighbors import BallTree
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
import zipfile, io

# **1. Load data & setup config**

In [180]:
# Load data
df_listings = pd.read_csv("../../datasets/hdb_active_listings_only.csv")
df_historical = pd.read_csv("../../datasets/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv")
df_amenities = pd.read_csv("../../datasets/hdb_amenity_table_full.csv")
df_trains = pd.read_csv("../../datasets/train_station_coords.csv")
df_bus = pd.read_csv("../../datasets/bus_stop_coords.csv")
df_schools = pd.read_csv("../../datasets/Generalinformationofschools_with_coords.csv")
df_polyclinics = pd.read_csv("../../datasets/singapore_polyclinics_with_coords.csv")
df_hawkers = pd.read_csv("../../datasets/hawker_centres_final.csv")
df_supermarkets = pd.read_csv("../../datasets/supermarkets_coords_clean.csv")
df_malls = pd.read_csv("../../datasets/sg_malls_final.csv")
df_rpi = pd.read_csv("../../datasets/HDBResalePriceIndex1Q2009100Quarterly.csv")
output_json = "../json_outputs/listings_predictions.json"
geocode_cache = "../json_outputs/geocode_cache.json"
model_paths = {
    "lgbm": "../models/lgb_model.joblib",
    "xgb": "../models/xgb_model.ubj",
    "cb": "../models/cb_model.cbm"
}
ci_offsets_json = "../json_outputs/ci_offsets.json"

SELECTED_MODEL = "ensemble_equal" # lgbm/xgb/cb/ensemble/ensemble_equal - we choose the best-performing model as per the DM tests in model_training.ipynb
ENSEMBLE_WEIGHTS = None

# Define constants
RPI_BASE = 100.0
CBD_LAT, CBD_LON = 1.2835, 103.8477 # Raffles Place MRT
HDB_ROOM_MAP = {
    "1-room": "1 ROOM",
    "2-room": "2 ROOM",
    "3-room": "3 ROOM",
    "4-room": "4 ROOM" # handle "5+-room" separately
}
FEATURES = [
    "town", "flat_type", "floor_area_sqm", "lease_commence_date",
    "remaining_lease", "lat", "lon",
    "mall_1_dist_m", "mall_2_dist_m", "mall_3_dist_m",
    "school_1_dist_m", "school_2_dist_m", "school_3_dist_m",
    "hawker_1_dist_m", "hawker_2_dist_m", "hawker_3_dist_m",
    "polyclinic_1_dist_m", "polyclinic_2_dist_m", "polyclinic_3_dist_m",
    "supermarket_1_dist_m", "supermarket_2_dist_m", "supermarket_3_dist_m",
    "train_1_dist_m", "train_2_dist_m", "train_3_dist_m",
    "bus_1_dist_m", "bus_2_dist_m", "bus_3_dist_m",
    "num_mrt_within_1km", "flag_mrt_within_500m",
    "num_primary_schools_within_1km", "num_hawkers_within_500m",
    "num_bus_within_400m",
    "dist_cbd",
    "month_index"
]

# **2. Geocoding current listings data**
* Use OneMap API to get `lat`, `lon`, `planning_area` (mapped to `town`)
* Save results to `outputs/geocode_cache.json` so that we only need to call the API once

In [181]:
# Helper functions
## Get coordinates of an address
def search_coords(search_val, token):
  search_url = f"https://www.onemap.gov.sg/api/common/elastic/search?searchVal={search_val}&returnGeom=Y&getAddrDetails=Y&pageNum=1"
  headers = {"Authorization": token}
  response = requests.get(search_url, headers=headers)
  response.raise_for_status()
  return response.json()

## From given address get lat, lon, full_address, postal
def get_address_info(address, token):
    search_result = search_coords(address, token)
    if not search_result.get("found") or search_result["found"] == 0:
        print(f"No results found for address: {address}")
        return None
    result = search_result["results"][0]
    postal = result.get("POSTAL")
    if postal == "NIL":
        postal = None
    return {
        "lat": float(result["LATITUDE"]),
        "lon": float(result["LONGITUDE"]),
        "full_address": result.get("ADDRESS"),
        "postal": postal
    }

## From given (lat, lon) get planning area
def get_planning_area(lat, lon, token):
  response = requests.get(
        "https://www.onemap.gov.sg/api/public/popapi/getPlanningarea",
        params={"latitude": lat, "longitude": lon},
        headers={"Authorization": token}
  )
  response.raise_for_status()
  data = response.json()
  if not data:
    return None
  return str(data[0].get("pln_area_n", "")).strip().upper()

## Add lat, lon, full_address, postal, town to listings
def geocode_listings(listings, token, cache_path):
  cache_dir = os.path.dirname(cache_path)
  if cache_dir:
    os.makedirs(cache_dir, exist_ok = True)

  if os.path.exists(cache_path):
    with open(cache_path) as f:
      cache = json.load(f)
  else:
    cache = {}

  results = []
  failed = []
  new_geocodes = 0

  for _, row in listings.iterrows():
    address = str(row["address"].strip())
    if address in cache:
      results.append(cache[address])
      continue
    entry = {
        "lat": None,
        "lon": None,
        "postal": None,
        "full_address": None,
        "town": None
    }

    try:
      address_info = get_address_info(address, token)
      if address_info is None:
        cache[address] = entry
        results.append(entry)
        new_geocodes += 1
        time.sleep(0.05)
        continue

      lat, lon = address_info["lat"], address_info["lon"] # first get address info
      postal = address_info["postal"]
      full_address = address_info["full_address"]
      time.sleep(0.05)

      town = get_planning_area(lat, lon, token) # then get town
      time.sleep(0.05)

      entry = {
          "lat": lat,
          "lon": lon,
          "postal": postal,
          "full_address": full_address,
          "town": town
      }
      new_geocodes += 1

    except Exception as e:
      print(f"Error geocoding address: {address}")
      print(e)

    cache[address] = entry
    results.append(entry)

  with open(cache_path, "w") as f:
    json.dump(cache, f, indent=2)
    print(f"Geocoding done. New API calls: {new_geocodes}. Cache saved to {cache_path}")

  listings = listings.copy()
  listings["lat"] = [r["lat"] for r in results]
  listings["lon"] = [r["lon"] for r in results]
  listings["postal"] = [r.get("postal") if r else None for r in results]
  listings["full_address"] = [r.get("full_address") if r else None for r in results]
  listings["town"] = [r.get("town") if r else None for r in results]

  failed = listings[
      listings["lat"].isna() |
      listings["lon"].isna() |
      listings["postal"].isna() |
      listings["full_address"].isna() |
      listings["town"].isna()
  ]
  for _, row in failed.iterrows():
    print("Failed to geocode:", row.get("address"))
  if len(failed):
    print(f"Failed to geocode {len(failed)} out of {len(listings)} addresses")

  return listings

In [182]:
df_listings = df_listings[df_listings["buildYear"].notna()]

# Geocode
load_dotenv(dotenv_path = "../.env")
auth_url = "https://www.onemap.gov.sg/api/auth/post/getToken"
ONEMAP_EMAIL = os.getenv("ONEMAP_EMAIL")
ONEMAP_PW = os.getenv("ONEMAP_PW")

payload = {
    "email": ONEMAP_EMAIL,
    "password": ONEMAP_PW
}

response = requests.request("POST", auth_url, json=payload)
token = response.json()["access_token"]

# Edit address for row with wrong addresses (verified manually)
address_corrections = {
    "Woodlands St.31": "301 Woodlands St 31"
}

for wrong, corrected in address_corrections.items():
    mask = df_listings["address"].str.strip() == wrong
    df_listings.loc[mask, "address"] = corrected
    print(f"Corrected {wrong} → {corrected} ({mask.sum()} row(s) affected)")

# Run API
df_listings = geocode_listings(df_listings, token, geocode_cache)
print("Listings after geocoding:", df_listings)

Corrected Woodlands St.31 → 301 Woodlands St 31 (1 row(s) affected)
Geocoding done. New API calls: 0. Cache saved to ../json_outputs/geocode_cache.json
Listings after geocoding:                             address  bathrooms  bedrooms    buildYear  \
0    451A Bukit Batok West Avenue 6          1         1  Built: 2018   
1            76 Telok Blangah Drive          1         1  Built: 1977   
2                  14 Taman Ho Swee          1         1  Built: 1970   
3    468A Bukit Batok West Avenue 9          1         1  Built: 2021   
4           180D Rivervale Crescent          1         1  Built: 2013   
..                              ...        ...       ...          ...   
931                 635A Senja Road          2         5  Built: 2014   
932         304 Woodlands Street 31          3         5  Built: 1993   
933         301 Woodlands Street 31          3         5  Built: 1993   
934          174 Lorong 1 Toa Payoh          2         6  Built: 1972   
935         163 Ang

In [183]:
# Clean the geocoded data
## Normalise towns
TOWN_NORMALISE = {
    "KALLANG": "KALLANG/WHAMPOA",
    "NOVENA": "CENTRAL AREA",
    "OUTRAM": "CENTRAL AREA",
    "ROCHOR": "CENTRAL AREA"
}
df_listings["town"] = df_listings["town"].replace(TOWN_NORMALISE)

## Drop row with ambiguous address: "Jalan Tenaga/ Bedok Reservoir Road/ Eunos Link"
df_listings = df_listings[df_listings["address"] != "Jalan Tenaga/ Bedok Reservoir Road/ Eunos Link"].reset_index(drop=True)

In [184]:
df_listings.to_csv("../csv_outputs/hdb_listings_combined_geocoded.csv")

# **3. Feature engineering**
* Standardise existing columns
  * Match column names to historical data
  * Parse `lease_commence_date`
  * Standardise `flat_type` categories
* Add distance features
  * Amenity distance features (join to amenities data)
  * Count/flag amenity features
  * `cbd_distance`
* Add temporal features
  * `month_index`
* Drop unnecessary columns

In [185]:
df_listings

,address,bathrooms,bedrooms,buildYear,floorAreaSqm,id,isAgentVerified,isVerifiedListing,listingFeatures/0,listingFeatures/1,...,row_index,listing_id,final_url,page_title,verdict,lat,lon,postal,full_address,town
0,451A Bukit Batok West Avenue 6,1,1,Built: 2018,38,60224822,False,False,1,1,...,0,60224822,https://www.propertyguru.com.sg/listing/hdb-fo...,451A Bukit Batok West Avenue 6 HDB Flat For Sa...,active_listing,1.353049,103.742933,651451,451A BUKIT BATOK WEST AVENUE 6 WEST TERRA @ BU...,BUKIT BATOK
1,76 Telok Blangah Drive,1,1,Built: 1977,44,500060411,False,False,1,1,...,1,500060411,https://www.propertyguru.com.sg/property-for-s...,76 Telok Blangah Drive for Sale in Singapore,active_listing,1.274384,103.808742,100076,76 TELOK BLANGAH DRIVE SINGAPORE 100076,BUKIT MERAH
2,14 Taman Ho Swee,1,1,Built: 1970,43,25428879,False,False,1,1,...,2,25428879,https://www.propertyguru.com.sg/listing/hdb-fo...,"14 Taman Ho Swee HDB Flat For Sale at S$ 328,8...",active_listing,1.287905,103.832198,161014,14 TAMAN HO SWEE BUKIT HO SWEE VIEW SINGAPORE ...,BUKIT MERAH
3,468A Bukit Batok West Avenue 9,1,1,Built: 2021,47,500065029,False,True,1,1,...,3,500065029,https://www.propertyguru.com.sg/property-for-s...,468A Bukit Batok West Avenue 9 for Sale in Sin...,active_listing,1.356495,103.742713,651468,468A BUKIT BATOK WEST AVENUE 9 WEST PLAINS @ B...,BUKIT BATOK
4,180D Rivervale Crescent,1,1,Built: 2013,47,60236988,False,False,1,1,...,4,60236988,https://www.propertyguru.com.sg/listing/hdb-fo...,180D Rivervale Crescent HDB Flat For Sale at S...,active_listing,1.389358,103.910398,544180,180D RIVERVALE CRESCENT RIVERVALE ARC SINGAPOR...,SENGKANG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
890,635A Senja Road,2,5,Built: 2014,112,500047152,False,False,5,2,...,1326,500047152,https://www.propertyguru.com.sg/listing/hdb-fo...,"635A Senja Road HDB Flat For Sale at S$ 870,00...",active_listing,1.386562,103.757208,671635,635A SENJA ROAD SENJA GATEWAY SINGAPORE 671635,BUKIT PANJANG
891,304 Woodlands Street 31,3,5,Built: 1993,163,500063526,False,False,5,3,...,1327,500063526,https://www.propertyguru.com.sg/listing/hdb-fo...,304 Woodlands Street 31 HDB Flat For Sale at S...,active_listing,1.429893,103.773854,730304,304 WOODLANDS STREET 31 FUCHUN NEIGHBOURHOOD C...,WOODLANDS
892,301 Woodlands Street 31,3,5,Built: 1993,163,500027859,False,False,5,3,...,1329,500027859,https://www.propertyguru.com.sg/listing/hdb-fo...,301 Woodlands Street 31 HDB Flat For Sale at S...,active_listing,1.431505,103.773516,730301,301 WOODLANDS STREET 31 FUCHUN NEIGHBOURHOOD C...,WOODLANDS
893,174 Lorong 1 Toa Payoh,2,6,Built: 1972,151,24344281,False,False,6,2,...,1330,24344281,https://www.propertyguru.com.sg/listing/hdb-fo...,174 Lorong 1 Toa Payoh HDB Flat For Sale at S$...,active_listing,1.330672,103.841918,310174,174 LORONG 1 TOA PAYOH TOA PAYOH GREEN SINGAPO...,TOA PAYOH


**3a. Standardise existing columns**

In [186]:
feature_df = df_listings.copy()

# Standardise column names
feature_df = feature_df.rename(
    columns = {
        "price": "asking_price",
        "floorAreaSqm": "floor_area_sqm",
        "id": "listing_id",
        "postedOn": "posted_on"
    }
)

# Parse buildYear into lease_commence_date
feature_df["lease_commence_date"] = (
    feature_df["buildYear"]
    .astype(str)
    .str.extract(r"(\d{4})")[0]
    .astype(int)
)

today = pd.Timestamp.today()
feature_df["remaining_lease"] = (feature_df["lease_commence_date"] + 99 - today.year).astype(float)

# Standardise flat_type categories
def map_flat_type(row):
  flat_type_cat = str(row.get("room_category", "")).strip().lower()
  if flat_type_cat in HDB_ROOM_MAP:
    return HDB_ROOM_MAP[flat_type_cat]
  if flat_type_cat == "5+-room":
    area = float(row.get("floor_area_sqm", 0))
    bed = int(row.get("bedrooms", 0))
    bath = int(row.get("bathrooms", 0))
    if bed == 4 and bath == 3:
      return "MULTI-GENERATION"
    elif area > 135:
      return "EXECUTIVE"
    else:
      return "5 ROOM"
  return flat_type_cat.upper()

feature_df["flat_type"] = feature_df.apply(map_flat_type, axis=1)

**3b. Distance features**

In [187]:
# Helper functions
def haversine_distance_vectorised(lat0, lon0, lats, lons):
    R = 6_371_000
    phi0, phi1 = np.radians(lat0), np.radians(lats)
    dphi = phi1 - phi0
    dlambda = np.radians(lons - lon0)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi0) * np.cos(phi1) * np.sin(dlambda / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))

def count_within_radius(block_lats, block_lons, amenity_lats, amenity_lons, radius_m):
  R = 6_371_000  # earth's radius in m

  # Reshape
  lat0 = np.radians(block_lats[:, None])
  lon0 = np.radians(block_lons[:, None])
  lat1 = np.radians(amenity_lats[None, :])
  lon1 = np.radians(amenity_lons[None, :])

  dlat = lat1 - lat0
  dlon = lon1 - lon0
  a = np.sin(dlat / 2) ** 2 + np.cos(lat0) * np.cos(lat1) * np.sin(dlon / 2) ** 2
  dist = R * 2 * np.arcsin(np.sqrt(a)) # shape: (n_blocks, n_amenities)

  return (dist <= radius_m).sum(axis=1).astype(int)

def count_within_radius_balltree(block_lats, block_lons, amenity_lats, amenity_lons, radius_m):
  R = 6_371_000
  amenity_coords = np.radians(
      np.column_stack([amenity_lats, amenity_lons])
  )
  block_coords = np.radians(
      np.column_stack([block_lats, block_lons])
  )
  tree = BallTree(amenity_coords, metric="haversine")
  counts = tree.query_radius(
      block_coords,
      r=radius_m / R, # Radians
      count_only=True
  )
  return counts.astype(int)

In [188]:
# Join to df_amenities to get distance features
## Split uppercased address into blk_no and street to match df_amenities join keys
feature_df["address_upper"] = feature_df["address"].str.strip().str.upper()
feature_df["blk_no_key"] = feature_df["address_upper"].str.split().str[0]
feature_df["street_key"] = feature_df["address_upper"].str.split(n=1).str[1]

## Normalise street names
street_abbrevs = {
    r"\bAVENUE\b": "AVE",
    r"\bCRESCENT\b": "CRES",
    r"\bDRIVE\b": "DR",
    r"\bSTREET\b": "ST",
    r"\bROAD\b": "RD",
    r"\bCLOSE\b": "CL",
    r"\bPLACE\b": "PL",
    r"\bBUKIT\b": "BT",
    r"\bNORTH\b": "NTH",
    r"\bSOUTH\b": "STH",
    r"\bJALAN\b": "JLN",
    r"\bLORONG\b": "LOR",
    r"\bCOMMONWEALTH\b": "C'WEALTH",
    r"\bGARDENS\b": "GDNS",
    r"\bCENTRAL\b": "CTRL",
    r"\bMARKET\b": "MKT",
    r"\bPARK\b": "PK",
    r"\bSAINT\b": "ST.",
    r"\bHEIGHTS\b": "HTS",
    r"\bUPPER\b": "UPP",
    r"\bKAMPONG\b": "KG",
    r"\bTERRACE\b": "TER"
}

for pattern, repl in street_abbrevs.items():
    feature_df["street_key"] = feature_df["street_key"].str.replace(pattern, repl, regex=True)

# Normalise to regular spaces
feature_df["street_key"] = (
    feature_df["street_key"]
    .str.replace(r"\s+", " ", regex=True) # collapses \xa0, \t, double-spaces into single space
    .str.strip()
)

In [189]:
# Join amenity data to get distance features
feature_df = feature_df.merge(
    df_amenities,
    left_on=["blk_no_key", "street_key"],
    right_on=["blk_no", "street"],
    how="left",
    suffixes=("", "_amenity")
)

## Drop helper columns used only for joining
feature_df = feature_df.drop(columns=["address_upper", "blk_no_key", "street_key", "blk_no", "street"])

## Check
unmatched = feature_df[feature_df["mall_1_dist_m"].isna()]
print(f"Rows with no amenity match: {len(unmatched)}")
print(unmatched[["address", "town"]].head(20))

Rows with no amenity match: 0
Empty DataFrame
Columns: [address, town]
Index: []


In [190]:
# Count/flag distance features
df_primary_schs = df_schools[df_schools["mainlevel_code"] == "PRIMARY"].dropna(subset=["lat", "long"])
df_trains = df_trains.dropna(subset=["lat", "lng"])
df_bus = df_bus.dropna(subset=["Latitude", "Longitude"])
df_hawkers = df_hawkers.dropna(subset=["lat", "lng"])

block_lats = feature_df["lat"].to_numpy()
block_lons = feature_df["lon"].to_numpy()

## num_mrt_within_1km
MRT_COUNT_THRESHOLD = 1000
feature_df["num_mrt_within_1km"] = count_within_radius(
    block_lats, block_lons, 
    df_trains["lat"].to_numpy(), df_trains["lng"].to_numpy(),
    MRT_COUNT_THRESHOLD
)

## flag_mrt_within_500m
MRT_FLAG_THRESHOLD = 500
feature_df["flag_mrt_within_500m"] = (
    count_within_radius(
        block_lats, block_lons,
        df_trains["lat"].to_numpy(), df_trains["lng"].to_numpy(),
        MRT_FLAG_THRESHOLD
    ) > 0
).astype(int)

## num_primary_schools_within_1km
SCHOOL_COUNT_THRESHOLD = 1000
feature_df["num_primary_schools_within_1km"] = count_within_radius(
    block_lats, block_lons,
    df_primary_schs["lat"].to_numpy(), df_primary_schs["long"].to_numpy(),
    SCHOOL_COUNT_THRESHOLD
)

## num_hawkers_within_500m
HAWKER_COUNT_THRESHOLD = 500
feature_df["num_hawkers_within_500m"] = count_within_radius(
    block_lats, block_lons,
    df_hawkers["lat"].to_numpy(), df_hawkers["lng"].to_numpy(),
    HAWKER_COUNT_THRESHOLD
)

## num_bus_within_400m
BUS_COUNT_THRESHOLD = 400
feature_df["num_bus_within_400m"] = count_within_radius_balltree(
    block_lats, block_lons,
    df_bus["Latitude"].to_numpy(), df_bus["Longitude"].to_numpy(),
    BUS_COUNT_THRESHOLD
)

COUNT_FEATURES = [
    "num_mrt_within_1km",
    "flag_mrt_within_500m",
    "num_bus_within_400m",
    "num_primary_schools_within_1km",
    "num_hawkers_within_500m"
]

## Check
print("Feature summary:")
print(feature_df[COUNT_FEATURES].describe().round(2))
print("Missing values:", feature_df[COUNT_FEATURES].isna().sum().to_dict())

Feature summary:
       num_mrt_within_1km  flag_mrt_within_500m  num_bus_within_400m  \
count              895.00                895.00               895.00   
mean                 2.33                  0.42                 9.67   
std                  2.19                  0.49                 3.24   
min                  0.00                  0.00                 1.00   
25%                  1.00                  0.00                 7.00   
50%                  2.00                  0.00                10.00   
75%                  3.00                  1.00                12.00   
max                 12.00                  1.00                20.00   

       num_primary_schools_within_1km  num_hawkers_within_500m  
count                          895.00                   895.00  
mean                             2.64                     0.71  
std                              1.42                     0.90  
min                              0.00                     0.00  
25%      

In [191]:
# Distance to CBD
feature_df["dist_cbd"] = haversine_distance_vectorised(CBD_LAT, CBD_LON, block_lats, block_lons)

**3c. Temporal features**

In [192]:
# Add month_index
feature_df["month_index"] = (today.year - 2017) * 12 + (today.month - 1)

In [193]:
# Standardise column data types
## Standardise data types
feature_df["postal"] = pd.to_numeric(feature_df["postal"], errors="coerce").astype("Int64")

categorical_cols = ["town", "flat_type"]
for col in categorical_cols:
    feature_df[col] = feature_df[col].astype("category")

## Check category values for categorical variables
for col in categorical_cols:
    print(f"Column: {col}")
    print(f"Total Unique Categories: {len(feature_df[col].cat.categories)}")
    print(feature_df[col].cat.categories.tolist())
    print("-" * 80)

# Drop unnecessary columns
DROP_COLS = [
    "row_index", "listing_id", 
    "final_url", "page_title",
    "posted_on", "buildYear",
    "bathrooms", "bedrooms",
    "isAgentVerified", "isVerifiedListing",
    "listingFeatures/0", "listingFeatures/1", "listingFeatures/2",
    "listingFeatures/3", "listingFeatures/4", "listingFeatures/5",
    "mrt/nearbyText",
    "thumbnail", "url",
    "pricePerAreaText", "pricePerSqm", "psfText",
    "propertyType", "recencyText", "tenure", "title",
    "room_category",
    "town_amenity", "postal_amenity", "full_address_amenity",
    "lat_amenity", "lon_amenity"
]
feature_df = feature_df.drop(columns=DROP_COLS)

# Check for any remaining NaN/null/NIL
null_mask = feature_df.isna() | feature_df.isin(["NIL", "null", "NULL", "None", ""])
null_counts = null_mask.sum()
null_counts[null_counts > 0]

Column: town
Total Unique Categories: 26
['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT BATOK', 'BUKIT MERAH', 'BUKIT PANJANG', 'BUKIT TIMAH', 'CENTRAL AREA', 'CHOA CHU KANG', 'CLEMENTI', 'GEYLANG', 'HOUGANG', 'JURONG EAST', 'JURONG WEST', 'KALLANG/WHAMPOA', 'MARINE PARADE', 'PASIR RIS', 'PUNGGOL', 'QUEENSTOWN', 'SEMBAWANG', 'SENGKANG', 'SERANGOON', 'TAMPINES', 'TOA PAYOH', 'WOODLANDS', 'YISHUN']
--------------------------------------------------------------------------------
Column: flat_type
Total Unique Categories: 5
['1 ROOM', '2 ROOM', '3 ROOM', '5 ROOM', 'EXECUTIVE']
--------------------------------------------------------------------------------


Series([], dtype: Int64)

In [194]:
feature_df.to_csv("../csv_outputs/hdb_listings_combined_features.csv")

# **4. Preparing outputs**
* Get current RPI
* Run models to get `predicted_price`
* Get confidence intervals of `predicted_price`: `ci_lower` and `ci_upper`
* Get recent median price of past transactions: `median_similar`
  * default: take last 6 months
  * if < 20 transactions of the same (town, flat_type) combination within the last 6 months, expand the window
  * some (town, flat_type) combinations are not represented at all in historical data --> returns NA
* Get `valuation_pct`

In [195]:
# RPI: get current quarter value
df_rpi.rename(columns={"index": "rpi"}, inplace=True)
df_rpi["rpi"] = pd.to_numeric(df_rpi["rpi"])
df_rpi.loc[len(df_rpi)] = { # Flash estimate from HDB
    "quarter": "2026-Q1", 
    "rpi": 203.4
 }

current_quarter = f"{today.year}-Q{((today.month - 1) // 3) + 1}"

if current_quarter not in df_rpi["quarter"].values:
  print(f"Current quarter {current_quarter} not in RPI data. Extrapolating...")

  recent = df_rpi.tail(4).copy().reset_index(drop=True)
  recent["t"] = range(4)
  slope, intercept = np.polyfit(recent["t"], recent["rpi"], deg=1)

  last_quarter = df_rpi["quarter"].iloc[-1]
  lq_year, lq_num = int(last_quarter.split("-Q")[0]), int(last_quarter.split("-Q")[1])
  q_year, q_num = int(current_quarter.split("-Q")[0]), int(current_quarter.split("-Q")[1])
  steps_ahead = (q_year - lq_year) * 4 + (q_num - lq_num)
  t_extrap = 3 + steps_ahead
  rpi_extrap = round(intercept + slope * t_extrap, 1)

  print(f"Extrapolated RPI for {current_quarter}: {rpi_extrap:.1f} (last known: {df_rpi["rpi"].iloc[-1]:.1f}, slope: {slope:+.2f}/quarter)")
  df_rpi = pd.concat(
      [df_rpi, pd.DataFrame([{"quarter": current_quarter, "rpi": rpi_extrap}])],
      ignore_index=True
  )
else:
  val = df_rpi.loc[df_rpi["quarter"] == current_quarter, "rpi"].iloc[0]
  print(f"RPI for {current_quarter}: {val:.1f}")

RPI_BASE = 100.0
RPI_CURRENT = df_rpi.loc[df_rpi["quarter"] == current_quarter, "rpi"].iloc[0]
print(f"Base RPI: {RPI_BASE}")
print(f"Current RPI: {RPI_CURRENT:.1f} (used to convert predictions back to nominal value)")

Current quarter 2026-Q2 not in RPI data. Extrapolating...
Extrapolated RPI for 2026-Q2: 203.8 (last known: 203.4, slope: +0.14/quarter)
Base RPI: 100.0
Current RPI: 203.8 (used to convert predictions back to nominal value)


In [196]:
# Load models
def load_models(model_name):
  models = {}
  if model_name in ("lgbm", "ensemble", "ensemble_equal"):
    print("Loading LightGBM model")
    with zipfile.ZipFile("../models/lgb_model.zip") as zf:
      with zf.open("lgb_model.joblib") as f:
        models["lgbm"] = joblib.load(io.BytesIO(f.read()))
  if model_name in ("xgb", "ensemble", "ensemble_equal"):
    print("Loading XGBoost model")
    m = xgb.XGBRegressor()
    m.load_model(model_paths["xgb"])
    models["xgb"] = m
  if model_name in ("cb", "ensemble", "ensemble_equal"):
    print("Loading CatBoost model")
    m = CatBoostRegressor()
    m.load_model(model_paths["cb"])
    models["cb"] = m
  return models

def predict(models, X):
  cat_cols = ["town", "flat_type"]

  def cb_predict(X):
    X_cb = X.copy()
    for col in cat_cols:
      X_cb[col] = X_cb[col].astype(str)
    pool = Pool(X_cb, cat_features=cat_cols)
    return models["cb"].predict(pool)

  if SELECTED_MODEL == "ensemble":
    w = ENSEMBLE_WEIGHTS
    if w is None:
      weights_path = "../models/ensemble_weights.npy"
      if os.path.exists(weights_path):
        w = np.load(weights_path)
      else:
        n = len(models)
        w = np.ones(n) / n
        print(f"Warning: ensemble_weights.npy not found at {weights_path}. Using equal weights {w}.")
    return (
        w[0] * cb_predict(X)
        + w[1] * models["xgb"].predict(X)
        + w[2] * models["lgbm"].predict(X)
    )
  
  if SELECTED_MODEL == "ensemble_equal":
    return np.stack([
        cb_predict(X),
        models["xgb"].predict(X),
        models["lgbm"].predict(X),
    ], axis=1).mean(axis=1)
  if SELECTED_MODEL == "cb":
    return np.array(cb_predict(X), dtype=float)
  return np.array(models[SELECTED_MODEL].predict(X), dtype=float)

models = load_models(SELECTED_MODEL)

Loading LightGBM model
Loading XGBoost model
Loading CatBoost model


In [197]:
# Predicted price
print(f"Loading model(s): {SELECTED_MODEL}")
X = feature_df[FEATURES].copy()
for col in ["town", "flat_type"]:
  X[col] = X[col].astype("category")

print(f"Running predictions on {len(X)} listings")
pred_real = predict(models, X)
pred_nominal = pred_real * (RPI_CURRENT / RPI_BASE)
print(f"Predicted {len(pred_real)} listings")

Loading model(s): ensemble_equal
Running predictions on 895 listings
Predicted 895 listings


In [198]:
# Confidence intervals
ci_lower_arr = ci_upper_arr = None
if os.path.exists(ci_offsets_json):
  with open(ci_offsets_json) as f:
    all_offsets = json.load(f)
  offsets = all_offsets.get(SELECTED_MODEL) or all_offsets.get("ensemble_equal")
  if offsets:
    ci_lower_arr = (pred_real + offsets["p025_real"]) * (RPI_CURRENT / RPI_BASE)
    ci_upper_arr = (pred_real + offsets["p975_real"]) * (RPI_CURRENT / RPI_BASE)
    key_used = SELECTED_MODEL if SELECTED_MODEL in all_offsets else "ensemble_equal"
    print(f"CI offsets applied from key {key_used}.")
else:
  print("ci_offsets.json not found, CI columns will be null.")

feature_df["pred_price_lower"] = ci_lower_arr if ci_lower_arr is not None else None
feature_df["pred_price_upper"] = ci_upper_arr if ci_upper_arr is not None else None

CI offsets applied from key ensemble_equal.


In [199]:
# Recent median (based on historical transactions, by town x flat_type)
MIN_SAMPLE = 20  # minimum transactions required, expand window if 6-month count falls below this
DEFAULT_MONTHS = 6

def compute_recent_median(historical, min_sample=MIN_SAMPLE, default_months=DEFAULT_MONTHS):
  """
  For each (town, flat_type):
  - Always use the full default_months window (e.g. 6 months)
  - If that window has fewer than min_sample transactions, expand backwards until min_sample is reached (or all data is exhausted)
  - Track number of samples taken for each median calculation 
  - Track actual months_back and flag if > 12
  """
  df = historical.copy()
  df["month"] = pd.to_datetime(df["month"])
  df = df.sort_values("month", ascending=False)

  today = pd.Timestamp.today()
  default_cutoff = today - pd.DateOffset(months=default_months)

  medians = {}
  months_back = {}
  sample_sizes = {}

  for (town, flat_type), group in df.groupby(["town", "flat_type"]):
    window = group[group["month"] >= default_cutoff]

    if len(window) >= min_sample:
      sample = window # enough in 6 months: use full window
    else:
      sample = group.head(min_sample) # too few: go further back until we get top min_sample

    medians[(town, flat_type)] = sample["resale_price"].median()
    sample_sizes[(town, flat_type)] = len(sample)

    oldest = sample["month"].min()
    n_months = (today.year - oldest.year) * 12 + (today.month - oldest.month)
    months_back[(town, flat_type)] = n_months

  return medians, months_back, sample_sizes

medians_recent, months_back, sample_sizes = compute_recent_median(df_historical)

feature_df["median_similar"] = feature_df.apply(lambda r: medians_recent.get((r["town"], r["flat_type"])), axis=1)
feature_df["median_months_back"] = feature_df.apply(lambda r: months_back.get((r["town"], r["flat_type"])), axis=1)
feature_df["median_sample_size"] = feature_df.apply(lambda r: sample_sizes.get((r["town"], r["flat_type"])), axis=1)
feature_df["median_old"] = feature_df["median_months_back"].apply(lambda x: bool(x > 12) if pd.notna(x) else None)

In [200]:
# Valuation
feature_df["predicted_price"] = pred_nominal.round(0)
feature_df["valuation_pct"] = (
    (feature_df["asking_price"] - feature_df["predicted_price"]) / feature_df["predicted_price"] * 100
).round(2)

In [201]:
# Assemble and write output
records = []
for i, row in feature_df.iterrows():
  ## Can edit to include more features depending on final frontend features
  record = {
      "address": row.get("address"),
      "town": row.get("town"),
      "flat_type": row.get("flat_type"),
      "floor_area_sqm": (float(row["floor_area_sqm"]) if pd.notna(row.get("floor_area_sqm")) else None),
      "lease_commence_date": (int(row["lease_commence_date"]) if pd.notna(row.get("lease_commence_date")) else None),
      "remaining_lease": (float(row["remaining_lease"]) if pd.notna(row.get("remaining_lease")) else None),
      "lat": float(row["lat"]),
      "lon": float(row["lon"]),
      "postal": (int(row["postal"]) if pd.notna(row.get("postal")) else None),
      "asking_price": (float(row["asking_price"]) if pd.notna(row.get("asking_price")) else None),
      "predicted_price": float(pred_nominal[i]),
      "predicted_price_lower": (float(row["pred_price_lower"]) if pd.notna(row.get("pred_price_lower")) else None),
      "predicted_price_upper": (float(row["pred_price_upper"]) if pd.notna(row.get("pred_price_upper")) else None),
      "valuation_pct": (float(row["valuation_pct"]) if pd.notna(row.get("valuation_pct")) else None),
      "median_similar": (float(row["median_similar"]) if pd.notna(row.get("median_similar")) else None),
      "median_months_back": (int(row["median_months_back"]) if pd.notna(row.get("median_months_back")) else None),
      "median_sample_size": (int(row["median_months_back"]) if pd.notna(row.get("median_sample_size")) else None),
      "median_old": row.get("median_old")
  }
  records.append(record)

with open(output_json, "w") as f:
  json.dump(records, f, indent=2, default=str)
print(f"Wrote {len(records)} records to {output_json}")

## Check
for r in records[:3]:
  print(f"{r["address"]} | pred = S${r["predicted_price"]} | ask = S${r["asking_price"]} | val_pct = {r["valuation_pct"]}")

Wrote 895 records to ../json_outputs/listings_predictions.json
451A Bukit Batok West Avenue 6 | pred = S$404598.5082755583 | ask = S$415000.0 | val_pct = 2.57
76 Telok Blangah Drive | pred = S$334368.75375443423 | ask = S$328888.0 | val_pct = -1.64
14 Taman Ho Swee | pred = S$332816.24269587797 | ask = S$328888.0 | val_pct = -1.18


In [202]:
feature_df.to_csv("../csv_outputs/hdb_listings_predictions.csv")

In [203]:
# Diagnostics/checking

## Predictions vs actual
over = (pred_nominal > feature_df["asking_price"].values).sum()
total = len(pred_nominal)
print(f"Predicted > actual: {over}/{total} ({over/total*100:.1f}%)")
under = (pred_nominal < feature_df["asking_price"].values).sum()
print(f"Actual > predicted: {under}/{total} ({under/total*100:.1f}%)")

## Range of valuation_pct (some outliers where valuation_pct > 100%)
print(feature_df["valuation_pct"].describe())

## Checking median transacted price
### Some listings have no previously transacted HDBs from the same town x flat type combination
print(sum(feature_df["median_similar"].isna()))
print(feature_df[feature_df["median_similar"].isna()][["town", "flat_type"]].value_counts())

## Checking columns with NAs
print(feature_df.isna().sum()[feature_df.isna().sum() > 0])

Predicted > actual: 276/895 (30.8%)
Actual > predicted: 619/895 (69.2%)
count    895.000000
mean       6.190749
std       25.546750
min      -24.220000
25%       -1.610000
50%        4.890000
75%       10.910000
max      530.550000
Name: valuation_pct, dtype: float64
176
town         flat_type
BUKIT BATOK  1 ROOM       27
SENGKANG     1 ROOM       27
SEMBAWANG    1 ROOM       17
ANG MO KIO   1 ROOM       16
PUNGGOL      1 ROOM       15
                          ..
TAMPINES     1 ROOM        0
             2 ROOM        0
             3 ROOM        0
             5 ROOM        0
             EXECUTIVE     0
Name: count, Length: 130, dtype: int64
median_similar        176
median_months_back    176
median_sample_size    176
median_old            176
dtype: int64
